In [ ]:
import pandas as pd


# Load your box score data
box_scores = pd.read_csv(
    "./data/player_box_standard_with_team.csv", dtype={56: str, 73: str}
)

# Rename columns if needed to match expected format
# For example:
column_mapping = {
    "points": "points",
    "field_goals_made": "field_goals_made",
    "field_goals_attempted": "field_goals_attempted",
    "three_point_field_goals_made": "three_point_field_goals_made",
    "three_point_field_goals_attempted": "three_point_field_goals_attempted",
    "free_throws_made": "free_throws_made",
    "free_throws_attempted": "free_throws_attempted",
    "offensive_rebounds": "offensive_rebounds",
    "defensive_rebounds": "defensive_rebounds",
    "rebounds": "rebounds",
    "assists": "assists",
    "steals": "steals",
    "blocks": "blocks",
    "turnovers": "turnovers",
    "fouls": "fouls",
    "plus_minus": "plus_minus",
}

box_scores = box_scores.rename(columns=column_mapping).query("season_type == 2")

# Initialize BPM calculator
box_scores

,game_id,season,season_type,game_date,game_date_time,athlete_id,athlete_display_name,team_id,team_name,team_location,...,opponent_team_location_team,opponent_team_name_team,opponent_team_abbreviation_team,opponent_team_display_name_team,opponent_team_short_display_name,opponent_team_color_team,opponent_team_alternate_color_team,opponent_team_logo_team,opponent_team_score_team,largest_lead
0,211030019,2002,2,2001-10-30,2001-10-31T00:30:00Z,353.0,Troy Hudson,19,Magic,Orlando,...,Toronto,Raptors,TOR,Toronto Raptors,Raptors,CE0F41,061922,https://a.espncdn.com/i/teamlogos/nba/500/tor.png,85.0,NaN
1,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,1026.0,Gerald Wallace,23,Kings,Sacramento,...,Seattle,SuperSonics,SEA,Seattle SuperSonics,SuperSonics,1C3F2C,f05133,NaN,95.0,NaN
2,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,376.0,Bobby Jackson,23,Kings,Sacramento,...,Seattle,SuperSonics,SEA,Seattle SuperSonics,SuperSonics,1C3F2C,f05133,NaN,95.0,NaN
3,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,259.0,Lawrence Funderburke,23,Kings,Sacramento,...,Seattle,SuperSonics,SEA,Seattle SuperSonics,SuperSonics,1C3F2C,f05133,NaN,95.0,NaN
4,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,793.0,Jabari Smith,23,Kings,Sacramento,...,Seattle,SuperSonics,SEA,Seattle SuperSonics,SuperSonics,1C3F2C,f05133,NaN,95.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810653,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,4431679.0,Precious Achiuwa,18,Knicks,New York,...,Brooklyn,Nets,BKN,Brooklyn Nets,Nets,000000,ffffff,https://a.espncdn.com/i/teamlogos/nba/500/bkn.png,105.0,9.0
810654,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,3147657.0,Mikal Bridges,18,Knicks,New York,...,Brooklyn,Nets,BKN,Brooklyn Nets,Nets,000000,ffffff,https://a.espncdn.com/i/teamlogos/nba/500/bkn.png,105.0,9.0
810655,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,3064230.0,Cameron Payne,18,Knicks,New York,...,Brooklyn,Nets,BKN,Brooklyn Nets,Nets,000000,ffffff,https://a.espncdn.com/i/teamlogos/nba/500/bkn.png,105.0,9.0
810656,401705753,2025,2,2025-04-13,2025-04-13T17:00:00Z,4576085.0,JD Davison,2,Celtics,Boston,...,Charlotte,Hornets,CHA,Charlotte Hornets,Hornets,008ca8,1d1060,https://a.espncdn.com/i/teamlogos/nba/500/cha.png,86.0,21.0


In [4]:
from bpm import BPMCalculator

bpm_calc = BPMCalculator()

# Calculate BPM metrics
bpm_results = bpm_calc.calculate_bpm(box_scores)

# Save results

print("BPM calculations complete. Results saved to player_bpm_results.csv")

BPM calculations complete. Results saved to player_bpm_results.csv


In [5]:
bpm_results["BPM*min"] = bpm_results["BPM"] * bpm_results["minutes"]
bpm_results["season_bpm"] = (
    bpm_results.groupby(["athlete_id", "season"])["BPM*min"]
    .transform("sum")
    .divide(bpm_results.groupby(["athlete_id", "season"])["minutes"].transform("sum"))
)

In [6]:
bpm_results.to_csv("./data/player_bpm_results.csv", index=False)

In [7]:
(
    bpm_results.loc[
        bpm_results.groupby(["athlete_id", "season"])["minutes"].transform("sum")
        >= 1000
    ]
    .sort_values(by="season_bpm", ascending=False)
    .drop_duplicates(subset=["athlete_id", "season"])
    .query("season == 2025")
    .head(20)[["season", "athlete_display_name", "season_bpm"]]
)

,season,athlete_display_name,season_bpm
754380,2025,Shai Gilgeous-Alexander,12.233478
753574,2025,Nikola Jokic,11.470893
748928,2025,Giannis Antetokounmpo,8.157904
756953,2025,Luka Doncic,7.870909
728669,2025,Jayson Tatum,7.164172
748525,2025,Tyrese Haliburton,6.762191
744884,2025,Donovan Mitchell,6.383048
739929,2025,Jimmy Butler III,6.331299
745170,2025,Evan Mobley,6.294171
739257,2025,Stephen Curry,6.249170
